# LeetCode 3661 — Maximum Walls Destroyed by Robots bằng Python

- Link: https://leetcode.com/problems/maximum-walls-destroyed-by-robots/
- Độ khó: **Hard**
- Chủ đề: Sắp xếp, **DP hai trạng thái** (hướng bắn), tìm nhị phân đếm điểm trong đoạn, xử lý **giao/hợp khoảng** để tránh đếm trùng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxWalls(self, robots: List[int], distance: List[int], walls: List[int]) -> int:
```

## 1. Mục tiêu học

1. Sắp robot theo vị trí: đạn của robot `i` bị chặn bởi robot `i−1` (bên trái) và `i+1` (bên phải) ⇒ chỉ tương tác với **hàng xóm**.
2. Đếm số tường trong một đoạn `[a, b]` bằng `bisect` trên mảng tường đã sắp.
3. Tránh **đếm trùng**: khe giữa robot `i` và `i+1` có thể bị cả `i` (bắn phải) và `i+1` (bắn trái) phủ ⇒ đếm **hợp** của hai khoảng.
4. DP `f[i][j]`: tốt nhất cho robot `0..i` biết hướng của robot `i+1` là `j` (theo doocs), hoặc DP "theo khe" với trạng thái là hướng của robot hiện tại.
5. Kiểm chứng bằng vét cạn `2^n` hướng bắn.

## 2. Đề bài & ví dụ

Mỗi robot bắn **một** viên đạn sang trái hoặc phải, đi tối đa `distance[i]`, phá mọi tường trên đường; đạn dừng khi chạm robot khác. Tường trùng vị trí robot bị chính robot đó phá. Tối đa hoá số tường **khác nhau** bị phá.

**Ví dụ 1:** `robots = [4], distance = [3], walls = [1,10]` → `1`.

**Ví dụ 2:** `robots = [10,2], distance = [5,1], walls = [5,2,7]` → `3`.

**Ví dụ 3:** `robots = [1,2], distance = [100,1], walls = [10]` → `0`.

**Ràng buộc:** `n, |walls| ≤ 10^5`.

**Quy ước mô hình hoá** (khớp lời giải tham khảo): đạn của robot ở `p` bắn trái phủ `[max(p − d, p_trái + 1), p]`, bắn phải phủ `[p, min(p + d, p_phải − 1)]` — tức là tường đặt **đúng tại vị trí robot khác** chỉ do robot đó phá.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
from functools import lru_cache
import random, time

examples = [([4], [3], [1, 10], 1), ([10, 2], [5, 1], [5, 2, 7], 3), ([1, 2], [100, 1], [10], 0)]
for r, d, w, exp in examples:
    print(r, d, w, "-> kỳ vọng", exp)

[4] [3] [1, 10] -> kỳ vọng 1
[10, 2] [5, 1] [5, 2, 7] -> kỳ vọng 3
[1, 2] [100, 1] [10] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Đếm điểm trong đoạn bằng `bisect`

Với `walls` đã sắp: số tường trong `[a, b]` = `bisect_right(walls, b) − bisect_left(walls, a)` (0 nếu `a > b`).

In [2]:
def count_in(walls, a, b):
    return 0 if a > b else bisect_right(walls, b) - bisect_left(walls, a)

W = sorted([5, 2, 7, 9])
print(count_in(W, 5, 10), count_in(W, 3, 4), count_in(W, 8, 1))

3 0 0


### 3.2. Hợp hai khoảng trên một khe

Khe giữa `p_i` và `p_{i+1}` (mở hai đầu): robot `i` bắn phải phủ `R = [p_i + 1, min(p_i + d_i, p_{i+1} − 1)]`, robot `i+1` bắn trái phủ `L = [max(p_{i+1} − d_{i+1}, p_i + 1), p_{i+1} − 1]`. Nếu cả hai cùng bắn vào khe, số tường bị phá = `|R ∪ L|` = `|R| + |L| − |R ∩ L|`. Vì `R` bám trái, `L` bám phải, `R ∩ L = [max(p_{i+1} − d_{i+1}, p_i+1), min(p_i + d_i, p_{i+1}−1)]`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tách tổng số tường bị phá thành:

1. **Tường tại vị trí robot**: luôn bị phá (bắn hướng nào cũng phủ vị trí của mình).
2. **Khe trái nhất** (trước robot đầu): chỉ robot 0 bắn trái mới phủ.
3. **Khe phải nhất**: chỉ robot cuối bắn phải mới phủ.
4. **Khe giữa robot `i` và `i+1`**: phụ thuộc cặp hướng `(dir_i, dir_{i+1})`: chỉ `i` bắn phải ⇒ `|R|`; chỉ `i+1` bắn trái ⇒ `|L|`; cả hai ⇒ `|R ∪ L|`; không ai ⇒ 0.

Mỗi khe chỉ phụ thuộc hướng của **hai robot kề** ⇒ DP theo thứ tự robot với trạng thái = hướng của robot hiện tại.

## 5. Cách 1 — Brute Force: mọi tổ hợp hướng

In [3]:
def brute(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); best = 0
    for mask in range(1 << n):
        hit = set()
        for i, (p, d) in enumerate(arr):
            if mask >> i & 1:                                  # bắn phải
                hi = p + d if i == n - 1 else min(p + d, arr[i + 1][0] - 1)
                lo = p
            else:                                              # bắn trái
                lo = p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)
                hi = p
            hit.update(w for w in walls if lo <= w <= hi)
        best = max(best, len(hit))
    return best

for r, d, w, exp in examples:
    print(brute(r, d, w), "kỳ vọng", exp)

1 kỳ vọng 1
3 kỳ vọng 3
0 kỳ vọng 0


**Nút thắt:** `2^n · |walls|`.

## 6. Ý tưởng thuật toán / bước đột phá

**DP theo doocs:** `f[i][j]` = số tường tối đa phá được bởi robot `0..i`, **biết** robot `i+1` bắn theo hướng `j` (0 = trái, 1 = phải). Khi robot `i` bắn phải và robot `i+1` bắn trái, cắt tầm phải của `i` tại `p_{i+1} − d_{i+1} − 1` để phần chung chỉ được đếm bởi robot `i+1`. Đáp án `f[n−1][1]` (robot "ảo" bên phải không chặn gì).

**DP theo khe:** `g[i][d]` = tốt nhất cho tường ở vị trí robot và các khe bên trái robot `i`, với robot `i` bắn hướng `d`; chuyển `g[i+1][d'] = max_d g[i][d] + gap(i, d, d')`.

## 7. Các cách tiếp theo

- **Cách 2:** đệ quy có nhớ `dfs(i, j)` (doocs).
- **Cách 3:** DP lặp `f[i][j]` (lời giải nộp).
- **Cách 4:** DP "theo khe" với công thức hợp khoảng — cách nhìn khác, tách rõ phần đếm trùng.

### 7.1. Cách 2 — Đệ quy có nhớ

In [4]:
def memo_dfs(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); W = sorted(walls)

    @lru_cache(None)
    def dfs(i, j):
        if i < 0: return 0
        p, d = arr[i]
        left = p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)
        best = dfs(i - 1, 0) + bisect_right(W, p) - bisect_left(W, left)
        right = p + d
        if i + 1 < n:
            right = min(right, arr[i + 1][0] - arr[i + 1][1] - 1) if j == 0 else min(right, arr[i + 1][0] - 1)
        best = max(best, dfs(i - 1, 1) + max(0, bisect_right(W, right) - bisect_left(W, p)))
        return best

    return dfs(n - 1, 1)

for r, d, w, exp in examples:
    print(memo_dfs(r, d, w), "kỳ vọng", exp)

1 kỳ vọng 1
3 kỳ vọng 3
0 kỳ vọng 0


**Lưu ý:** với `n = 10^5`, đệ quy sâu `10^5` vượt giới hạn mặc định của Python ⇒ cần bản lặp (Cách 3).

### 7.2. Cách 3 — DP lặp (lời giải chính)

In [5]:
def dp_iterative(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); W = sorted(walls)
    prev = [0, 0]                                         # f[i-1][0], f[i-1][1]
    for i in range(n):
        p, d = arr[i]
        left = p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)
        cntL = bisect_right(W, p) - bisect_left(W, left)
        cur = [0, 0]
        for j in range(2):
            right = p + d
            if i + 1 < n:
                right = min(right, arr[i + 1][0] - arr[i + 1][1] - 1) if j == 0 else min(right, arr[i + 1][0] - 1)
            cntR = max(0, bisect_right(W, right) - bisect_left(W, p))
            cur[j] = max(prev[0] + cntL, prev[1] + cntR)
        prev = cur
    return prev[1]

for r, d, w, exp in examples:
    print(dp_iterative(r, d, w), "kỳ vọng", exp)

1 kỳ vọng 1
3 kỳ vọng 3
0 kỳ vọng 0


### 7.3. Cách 4 — DP theo khe

In [6]:
def dp_gaps(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); W = sorted(walls)
    cnt = lambda a, b: count_in(W, a, b)
    own = sum(cnt(p, p) for p, _ in arr)                  # tường tại vị trí robot: luôn bị phá
    p0, d0 = arr[0]
    g = [cnt(p0 - d0, p0 - 1), 0]                          # khe trái nhất: chỉ khi robot 0 bắn trái (d = 0)
    for i in range(n - 1):
        (p, d), (q, e) = arr[i], arr[i + 1]
        R = (p + 1, min(p + d, q - 1)); L = (max(q - e, p + 1), q - 1)
        r_cnt = cnt(*R); l_cnt = cnt(*L)
        both = r_cnt + l_cnt - cnt(max(R[0], L[0]), min(R[1], L[1]))
        gap = {(0, 0): l_cnt, (0, 1): 0, (1, 0): both, (1, 1): r_cnt}     # (hướng i, hướng i+1), 1 = phải
        g = [max(g[a] + gap[(a, b)] for a in range(2)) for b in range(2)]
    pn, dn = arr[-1]
    g[1] += cnt(pn + 1, pn + dn)                          # khe phải nhất: chỉ khi robot cuối bắn phải
    return own + max(g)

for r, d, w, exp in examples:
    print(dp_gaps(r, d, w), "kỳ vọng", exp)

1 kỳ vọng 1
3 kỳ vọng 3
0 kỳ vọng 0


## 8. TRACE từng bước

In [7]:
def trace(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); W = sorted(walls)
    print("robot (vị trí, tầm) đã sắp:", arr, "| tường:", W)
    cnt = lambda a, b: count_in(W, a, b)
    own = sum(cnt(p, p) for p, _ in arr); print("tường tại vị trí robot:", own)
    p0, d0 = arr[0]; g = [cnt(p0 - d0, p0 - 1), 0]
    print(f"khe trái nhất [{p0 - d0}, {p0 - 1}]: g = {g} (g[0] = robot 0 bắn trái, g[1] = bắn phải)")
    for i in range(n - 1):
        (p, d), (q, e) = arr[i], arr[i + 1]
        R = (p + 1, min(p + d, q - 1)); L = (max(q - e, p + 1), q - 1)
        r_cnt, l_cnt = cnt(*R), cnt(*L); both = r_cnt + l_cnt - cnt(max(R[0], L[0]), min(R[1], L[1]))
        gap = {(0, 0): l_cnt, (0, 1): 0, (1, 0): both, (1, 1): r_cnt}
        g = [max(g[a] + gap[(a, b)] for a in range(2)) for b in range(2)]
        print(f"khe ({p},{q}): R={R} có {r_cnt}, L={L} có {l_cnt}, hợp={both} -> g = {g}")
    pn, dn = arr[-1]; g[1] += cnt(pn + 1, pn + dn)
    print(f"khe phải nhất: g = {g}; đáp án = {own} + {max(g)} = {own + max(g)} | brute: {brute(robots, distance, walls)}")

trace([10, 2], [5, 1], [5, 2, 7])

robot (vị trí, tầm) đã sắp: [(2, 1), (10, 5)] | tường: [2, 5, 7]
tường tại vị trí robot: 1
khe trái nhất [1, 1]: g = [0, 0] (g[0] = robot 0 bắn trái, g[1] = bắn phải)
khe (2,10): R=(3, 3) có 0, L=(5, 9) có 2, hợp=2 -> g = [2, 0]
khe phải nhất: g = [2, 0]; đáp án = 1 + 2 = 3 | brute: 3


In [8]:
trace([3, 8, 12], [4, 3, 5], [1, 4, 5, 6, 7, 8, 10, 11, 15, 16])

robot (vị trí, tầm) đã sắp: [(3, 4), (8, 3), (12, 5)] | tường: [1, 4, 5, 6, 7, 8, 10, 11, 15, 16]
tường tại vị trí robot: 1
khe trái nhất [-1, 2]: g = [1, 0] (g[0] = robot 0 bắn trái, g[1] = bắn phải)
khe (3,8): R=(4, 7) có 4, L=(5, 7) có 3, hợp=4 -> g = [4, 4]
khe (8,12): R=(9, 11) có 2, L=(9, 11) có 2, hợp=2 -> g = [6, 6]
khe phải nhất: g = [6, 8]; đáp án = 1 + 8 = 9 | brute: 9


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxWalls(self, robots: List[int], distance: List[int], walls: List[int]) -> int:
        arr = sorted(zip(robots, distance))
        n = len(arr)
        W = sorted(walls)
        f0 = f1 = 0                                       # f[i-1][0], f[i-1][1]
        for i in range(n):
            p, d = arr[i]
            left = p - d
            if i:
                left = max(left, arr[i - 1][0] + 1)
            cntL = bisect_right(W, p) - bisect_left(W, left)
            lo = bisect_left(W, p)
            res = [0, 0]
            for j in range(2):
                right = p + d
                if i + 1 < n:
                    q, e = arr[i + 1]
                    right = min(right, q - e - 1) if j == 0 else min(right, q - 1)
                cntR = max(0, bisect_right(W, right) - lo)
                res[j] = max(f0 + cntL, f1 + cntR)
            f0, f1 = res
        return f1

sol = Solution()
for r, d, w, exp in examples:
    got = sol.maxWalls(r, d, w)
    print(got, "✔" if got == exp else "✘")

1 ✔
3 ✔
0 ✔


## 10. Vì sao thuật toán đúng

1. **Cục bộ:** sau khi sắp, phạm vi đạn của robot `i` nằm trong `(p_{i−1}, p_{i+1})` (cộng chính `p_i`). Tập tường bị phá là hợp của các phạm vi; tường ở khe `(p_i, p_{i+1})` chỉ có thể bị robot `i` hoặc `i+1` phá.
2. **Cách 3/2 (doocs):** khi robot `i+1` bắn trái (`j = 0`), robot `i` bắn phải chỉ được tính phần **không** nằm trong tầm trái của `i+1` (cắt tại `p_{i+1} − d_{i+1} − 1`) — phần chung được tính một lần ở robot `i+1`. Khi `i+1` bắn phải, robot `i` được tính tới `p_{i+1} − 1`. Vì `f[i][j]` phụ thuộc đúng hướng của `i+1`, quy hoạch động duyệt đủ `2^n` tổ hợp một cách gộp.
3. **Cách 4:** tổng = tường tại robot (luôn bị phá) + khe trái nhất + Σ khe giữa + khe phải nhất; mỗi khe phụ thuộc đúng hai hướng kề nhau, và `|R ∪ L| = |R| + |L| − |R ∩ L|` với `R ∩ L` là đoạn (vì `R` bám trái, `L` bám phải của cùng khe).
4. Cả hai DP khớp vét cạn `2^n` trên dữ liệu ngẫu nhiên (mục 12).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n · |walls|)` | `O(|walls|)` |
| 2. Đệ quy có nhớ | `O((n + m) log m)` | `O(n)` ngăn xếp |
| 3. DP lặp | `O(n log n + m log m)` | `O(1)` ngoài sắp xếp |
| 4. DP theo khe | `O(n log n + m log m)` | `O(1)` ngoài sắp xếp |

## 12. Bộ test

In [10]:
METHODS = [brute, memo_dfs, dp_iterative, dp_gaps, lambda r, d, w: Solution().maxWalls(r, d, w)]
for r, d, w, exp in examples:
    assert all(f(r, d, w) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().maxWalls([5], [1], [5]) == 1                       # tường trùng vị trí robot
assert Solution().maxWalls([5, 6], [10, 10], [5, 6]) == 2
assert Solution().maxWalls([1, 3], [5, 5], [2]) == 1                  # cả hai cùng phủ, chỉ đếm 1
assert Solution().maxWalls([1, 100], [1, 1], [50]) == 0
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3661)
for _ in range(2000):
    n = rng.randint(1, 7)
    robots = rng.sample(range(1, 40), n); distance = [rng.randint(1, 10) for _ in range(n)]
    walls = rng.sample(range(1, 45), rng.randint(1, 15))
    b = brute(robots, distance, walls)
    assert all(f(robots, distance, walls) == b for f in METHODS[1:]), (robots, distance, walls)
print("2000 test ngẫu nhiên: OK")

2000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn
rng = random.Random(1)
robots = rng.sample(range(1, 10**9), 10**5); distance = [rng.randint(1, 10**5) for _ in range(10**5)]
walls = rng.sample(range(1, 10**9), 10**5)
t0 = time.time(); x = Solution().maxWalls(robots, distance, walls); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp DP theo khe:", x == dp_gaps(robots, distance, walls))

đáp án: 98506 | thời gian: 0.26 s | khớp DP theo khe: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `|R| + |L|` khi cả hai cùng bắn vào một khe (đếm trùng)

In [14]:
def wrong_double_count(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); W = sorted(walls)
    cnt = lambda a, b: count_in(W, a, b)
    own = sum(cnt(p, p) for p, _ in arr); p0, d0 = arr[0]; g = [cnt(p0 - d0, p0 - 1), 0]
    for i in range(n - 1):
        (p, d), (q, e) = arr[i], arr[i + 1]
        r_cnt = cnt(p + 1, min(p + d, q - 1)); l_cnt = cnt(max(q - e, p + 1), q - 1)
        gap = {(0, 0): l_cnt, (0, 1): 0, (1, 0): r_cnt + l_cnt, (1, 1): r_cnt}        # SAI
        g = [max(g[a] + gap[(a, b)] for a in range(2)) for b in range(2)]
    pn, dn = arr[-1]; g[1] += cnt(pn + 1, pn + dn)
    return own + max(g)

print("sai:", wrong_double_count([1, 3], [5, 5], [2]), "| đúng:", brute([1, 3], [5, 5], [2]))

sai: 2 | đúng: 1


### Lỗi 2 — Quên robot chặn đạn

In [15]:
def wrong_no_block(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); best = 0
    for mask in range(1 << n):
        hit = set()
        for i, (p, d) in enumerate(arr):
            lo, hi = (p, p + d) if mask >> i & 1 else (p - d, p)                     # SAI: không cắt tại robot kề
            hit.update(w for w in walls if lo <= w <= hi)
        best = max(best, len(hit))
    return best

print("sai:", wrong_no_block([1, 2], [100, 1], [10]), "| đúng:", brute([1, 2], [100, 1], [10]))

sai: 1 | đúng: 0


### Lỗi 3 — Không sắp xếp robot theo vị trí

In [16]:
def wrong_unsorted(robots, distance, walls):
    arr = list(zip(robots, distance)); n = len(arr); W = sorted(walls); prev = [0, 0]     # SAI: thiếu sort
    for i in range(n):
        p, d = arr[i]; left = p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)
        cntL = max(0, bisect_right(W, p) - bisect_left(W, left)); cur = [0, 0]
        for j in range(2):
            right = p + d
            if i + 1 < n: right = min(right, arr[i + 1][0] - arr[i + 1][1] - 1) if j == 0 else min(right, arr[i + 1][0] - 1)
            cur[j] = max(prev[0] + cntL, prev[1] + max(0, bisect_right(W, right) - bisect_left(W, p)))
        prev = cur
    return prev[1]

rng = random.Random(12)
while True:
    n = rng.randint(2, 5); r = rng.sample(range(1, 30), n); d = [rng.randint(1, 8) for _ in range(n)]
    w = rng.sample(range(1, 35), rng.randint(1, 10))
    if wrong_unsorted(r, d, w) != brute(r, d, w): break
print("phản ví dụ:", r, d, w, "| sai:", wrong_unsorted(r, d, w), "| đúng:", brute(r, d, w))

phản ví dụ: [22, 20, 16, 13, 18] [1, 2, 4, 5, 6] [25, 20, 8, 9, 32, 11] | sai: 1 | đúng: 4


### Lỗi 4 — Đệ quy sâu `10^5` trong Python

`memo_dfs` với `n = 10^5` sẽ `RecursionError` — dùng DP lặp.

In [17]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit(), "< n tối đa 10^5")

giới hạn đệ quy mặc định: 1000 < n tối đa 10^5


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mỗi phần tử chọn một trong vài hướng, và chỉ tương tác với **hàng xóm** sau khi sắp ⇒ DP với trạng thái = lựa chọn của phần tử hiện tại/kế tiếp.
- Đếm điểm trong đoạn trên tập tĩnh ⇒ sắp + `bisect`.
- Hai nguồn cùng phủ một vùng ⇒ tính **hợp** hoặc gán phần chung cho đúng một nguồn.

## 15. Biến thể của bài

1. **Mỗi robot bắn cả hai hướng**: không còn lựa chọn — tổng hợp các khoảng.
2. **Đạn xuyên qua robot**: các khoảng chồng nhau tuỳ ý ⇒ cần quy hoạch trên các điểm cuối (khó hơn).
3. **Mỗi tường có trọng số**: thay `bisect` bằng tổng tiền tố trọng số.
4. **Tối thiểu** số tường bị phá: đổi `max` thành `min`.

In [18]:
# Biến thể 3: tường có trọng số — tổng tiền tố theo vị trí đã sắp
def max_weight(robots, distance, walls, weight):
    arr = sorted(zip(robots, distance)); n = len(arr)
    order = sorted(range(len(walls)), key=lambda i: walls[i]); W = [walls[i] for i in order]
    pre = [0]
    for i in order: pre.append(pre[-1] + weight[i])
    s = lambda a, b: 0 if a > b else pre[bisect_right(W, b)] - pre[bisect_left(W, a)]
    own = sum(s(p, p) for p, _ in arr); p0, d0 = arr[0]; g = [s(p0 - d0, p0 - 1), 0]
    for i in range(n - 1):
        (p, d), (q, e) = arr[i], arr[i + 1]
        R = (p + 1, min(p + d, q - 1)); L = (max(q - e, p + 1), q - 1)
        rs, ls = s(*R), s(*L); both = rs + ls - s(max(R[0], L[0]), min(R[1], L[1]))
        gap = {(0, 0): ls, (0, 1): 0, (1, 0): both, (1, 1): rs}
        g = [max(g[a] + gap[(a, b)] for a in range(2)) for b in range(2)]
    pn, dn = arr[-1]; g[1] += s(pn + 1, pn + dn)
    return own + max(g)

def brute_weight(robots, distance, walls, weight):
    arr = sorted(zip(robots, distance)); n = len(arr); best = 0
    for mask in range(1 << n):
        hit = set()
        for i, (p, d) in enumerate(arr):
            if mask >> i & 1: lo, hi = p, (p + d if i == n - 1 else min(p + d, arr[i + 1][0] - 1))
            else: lo, hi = (p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)), p
            hit.update(j for j, w in enumerate(walls) if lo <= w <= hi)
        best = max(best, sum(weight[j] for j in hit))
    return best

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 6); r = rng.sample(range(1, 30), n); d = [rng.randint(1, 8) for _ in range(n)]
    w = rng.sample(range(1, 35), rng.randint(1, 10)); wt = [rng.randint(1, 9) for _ in w]
    assert max_weight(r, d, w, wt) == brute_weight(r, d, w, wt)
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | `2^n` hướng | `O(2^n n m)` | Kiểm chứng |
| 2. Đệ quy có nhớ | `dfs(i, hướng i+1)` | `O((n+m) log m)` | `n` nhỏ (đệ quy) |
| 3. DP lặp | Như 2, không đệ quy | `O((n+m) log)` | **Lời giải chuẩn** |
| 4. DP theo khe | Hợp khoảng mỗi khe | `O((n+m) log)` | Dễ mở rộng (trọng số) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `robots = [5], distance = [2], walls = [3, 4, 6, 7, 8]`.

**Bài 2 (tính tay).** `robots = [2, 6], distance = [3, 3], walls = [1, 3, 4, 5, 8]`.

**Bài 3 (điền TODO).** Viết `min_walls(robots, distance, walls)` — số tường **ít nhất** bị phá.

**Bài 4 (tự viết lại).** Trả về luôn **hướng bắn** của mỗi robot (theo thứ tự đầu vào) đạt tối ưu.

**Bài 5 (mini challenge).** Mỗi robot có **hai** viên đạn (bắn được cả hai hướng hoặc hai lần một hướng — vô ích). Tính số tường bị phá tối đa.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_walls(robots, distance, walls):
    # TODO: DP theo khe với min
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def two_bullets(robots, distance, walls):
    # TODO: mỗi robot phủ cả hai phía (bị chặn bởi hàng xóm); đếm hợp
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Bắn trái phủ `[3, 5]`: tường `3, 4`. (2) Bắn phải phủ `[5, 7]`: `6, 7`. (3) Đáp án `2`.

**Bài 2.** (1) Robot `(2, 3)` và `(6, 3)`; khe trái nhất `[−1, 1]` có tường `1`, khe giữa `(2, 6)` có `3, 4, 5`, khe phải nhất `[7, 9]` có `8`. (2) Robot 0 bắn phải hoặc robot 1 bắn trái đều phá cả `3, 4, 5`. (3) Tốt nhất: robot 0 trái (`1`) + robot 1 trái (`3,4,5`) = 4, hoặc robot 0 phải (`3,4,5`) + robot 1 phải (`8`) = 4 ⇒ `4`.

**Bài 3.** (1) Như DP theo khe. (2) Đổi `max` → `min`. (3) Kiểm chứng vét cạn.

**Bài 4.** (1) Lưu cha trong DP theo khe. (2) Truy vết từ trạng thái cuối. (3) Ánh xạ về thứ tự đầu vào.

**Bài 5.** (1) Mỗi khe bị phủ bởi `R ∪ L` luôn. (2) Cộng tường tại robot, hai khe biên. (3) Không cần DP.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().maxWalls([5], [2], [3, 4, 6, 7, 8]) == 2
assert Solution().maxWalls([2, 6], [3, 3], [1, 3, 4, 5, 8]) == 4 == brute([2, 6], [3, 3], [1, 3, 4, 5, 8])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def min_walls_sol(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); W = sorted(walls)
    cnt = lambda a, b: count_in(W, a, b)
    own = sum(cnt(p, p) for p, _ in arr); p0, d0 = arr[0]; g = [cnt(p0 - d0, p0 - 1), 0]
    for i in range(n - 1):
        (p, d), (q, e) = arr[i], arr[i + 1]
        R = (p + 1, min(p + d, q - 1)); L = (max(q - e, p + 1), q - 1)
        rc, lc = cnt(*R), cnt(*L); both = rc + lc - cnt(max(R[0], L[0]), min(R[1], L[1]))
        gap = {(0, 0): lc, (0, 1): 0, (1, 0): both, (1, 1): rc}
        g = [min(g[a] + gap[(a, b)] for a in range(2)) for b in range(2)]
    pn, dn = arr[-1]; g[1] += cnt(pn + 1, pn + dn)
    return own + min(g)

def best_directions(robots, distance, walls):
    idx = sorted(range(len(robots)), key=lambda i: robots[i]); arr = [(robots[i], distance[i]) for i in idx]
    n = len(arr); W = sorted(walls); cnt = lambda a, b: count_in(W, a, b)
    p0, d0 = arr[0]; g = [cnt(p0 - d0, p0 - 1), 0]; par = []
    for i in range(n - 1):
        (p, d), (q, e) = arr[i], arr[i + 1]
        R = (p + 1, min(p + d, q - 1)); L = (max(q - e, p + 1), q - 1)
        rc, lc = cnt(*R), cnt(*L); both = rc + lc - cnt(max(R[0], L[0]), min(R[1], L[1]))
        gap = {(0, 0): lc, (0, 1): 0, (1, 0): both, (1, 1): rc}
        ng = []; pp = []
        for b in range(2):
            a = max(range(2), key=lambda a: g[a] + gap[(a, b)]); ng.append(g[a] + gap[(a, b)]); pp.append(a)
        g = ng; par.append(pp)
    pn, dn = arr[-1]; g[1] += cnt(pn + 1, pn + dn)
    b = max(range(2), key=lambda b: g[b]); dirs = [b]
    for pp in reversed(par): b = pp[b]; dirs.append(b)
    dirs.reverse(); res = [None] * n
    for k, i in enumerate(idx): res[i] = "R" if dirs[k] else "L"
    return res

def simulate(robots, distance, walls, dirs):
    arr = sorted(zip(robots, distance, dirs)); n = len(arr); hit = set()
    for i, (p, d, c) in enumerate(arr):
        if c == "R": lo, hi = p, (p + d if i == n - 1 else min(p + d, arr[i + 1][0] - 1))
        else: lo, hi = (p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)), p
        hit.update(w for w in walls if lo <= w <= hi)
    return len(hit)

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 6); r = rng.sample(range(1, 30), n); d = [rng.randint(1, 8) for _ in range(n)]
    w = rng.sample(range(1, 35), rng.randint(1, 10))
    dirs = best_directions(r, d, w)
    assert simulate(r, d, w, dirs) == brute(r, d, w)
    exp_min = min(simulate(r, d, w, [("R" if m >> k & 1 else "L") for k in range(n)]) for m in range(1 << n))
    assert min_walls_sol(r, d, w) == exp_min
print("Bài 3, 4 OK; ví dụ 2:", best_directions([10, 2], [5, 1], [5, 2, 7]))

Bài 3, 4 OK; ví dụ 2: ['L', 'L']


In [23]:
# Bài 5: mỗi robot bắn cả hai hướng
def two_bullets_sol(robots, distance, walls):
    return simulate_both(robots, distance, walls)

def simulate_both(robots, distance, walls):
    arr = sorted(zip(robots, distance)); n = len(arr); hit = set()
    for i, (p, d) in enumerate(arr):
        lo = p - d if i == 0 else max(p - d, arr[i - 1][0] + 1)
        hi = p + d if i == n - 1 else min(p + d, arr[i + 1][0] - 1)
        hit.update(w for w in walls if lo <= w <= hi)
    return len(hit)

print("Bài 5:", two_bullets_sol([10, 2], [5, 1], [5, 2, 7]), "| ví dụ 3:", two_bullets_sol([1, 2], [100, 1], [10]))
assert two_bullets_sol([4], [3], [1, 10]) == 1 and two_bullets_sol([4], [6], [1, 10]) == 2

Bài 5: 3 | ví dụ 3: 0


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
arr = sorted(zip(robots, distance)); W = sorted(walls)
f0 = f1 = 0
for i, (p, d) in enumerate(arr):
    left = max(p - d, p_trái + 1);  cntL = #W trong [left, p]
    for j in (0, 1):  # hướng của robot i+1
        right = min(p + d, (q - e - 1) nếu j == 0 else (q - 1))
        res[j] = max(f0 + cntL, f1 + #W trong [p, right])
    f0, f1 = res
return f1
```

**Checklist:**

- [ ] Sắp robot và tường?
- [ ] Đạn bị chặn tại robot kề (`p_kề ± 1`)?
- [ ] Phần chung của một khe chỉ đếm một lần?
- [ ] Không đệ quy sâu?
- [ ] Đã so với vét cạn `2^n`?